# Data organisation

This section considers the organisation of the data in ClickHouse.

## Tables

Here are some useful ClickHouse features for checking table properties:

- `DESCRITBE TABLE` command: to check the columns and their types of the table.
- `system.tables` table: stores information about tables in clickhouse.
- `system.columns`table: stores information about available columns from all tables.

### Describe

Use the `TABLE DESCRIBE` command to view the properties of the table. This is practcally useful for identifing the names and types of the columns.

Check more in the page [DESCRIBE TABLE](https://clickhouse.com/docs/sql-reference/statements/describe-table).

---

The following cell shows the example of `DESCRIBE TABLE` usage.

In [3]:
--ClickHouse
DESCRIBE TABLE
SELECT *
FROM VALUES(
    'id UInt32, name String',
    (1, 'Alice'),
    (2, 'Bob'),
    (3, 'Carol')
);

name,type,default_type,default_expression,comment,codec_expression,ttl_expression
id,UInt32,,,,,
name,String,,,,,


## Databases

Most essences in clickhouse are separated by databases. The database can contain: tables, views, dictionaries and functions.

Important commands to handle databases:

- `SHOW DATABASES`.
- `CREATE DATABASE <name>`
- `DROP DATABASE <name>`.
- `USE DATABASE`: to select the context, all subsequent queries will automatilly will refer to the essesnces of the specific database.

Check more in [Databases](data_organisation/databases.ipynb) page.

---

The following cell show the initial list of available databases.

In [2]:
--ClickHouse
SHOW DATABASES;

name
INFORMATION_SCHEMA
default
information_schema
system


After `CREATE DATABASE` there is a new database with the corresponding name:

In [5]:
--ClickHouse
CREATE DATABASE test_database;
SHOW DATABASES;

name
INFORMATION_SCHEMA
default
information_schema
system
test_database


After `DROP DATABASE` the database is removed.

In [6]:
--ClickHouse
DROP DATABASE test_database;
SHOW DATABASES;

name
INFORMATION_SCHEMA
default
information_schema
system


## Materialized view

A materialized view (MV) is a special type of object that uses a chosen query to periodically update some data. In ClickHouse there are two types of MVs:

- **Incremental** adds the information written to the source tables to the target table.
- **Refreshable** works as a cron job, that automatically rerunning the query and rewriting the information in the target.

For more check:

- The [Materialized views](https://clickhouse.com/docs/materialized-views) section of the documentation.
- The [Materialized view](data_organisation/materialized_view.ipynb) page.

---

The following cell defines the `source` table and materialized view (it's incremental by default) that selects only data with `category = 'target'`.

In [28]:
--ClickHouse
DROP TABLE IF EXISTS data_source;
DROP TABLE IF EXISTS target_category;

CREATE TABLE data_source
(
    id Int8,
    category TEXT
);

CREATE MATERIALIZED VIEW target_category AS
SELECT * FROM data_source WHERE category='target';

read_rows,read_bytes,written_rows,written_bytes,total_rows_to_read,result_rows,result_bytes,elapsed_ns,memory_usage,query_id
0,0,0,0,0,0,0,309631,1050807,0eeda369-e4ee-42be-9f8f-a492131c9b2a


read_rows,read_bytes,written_rows,written_bytes,total_rows_to_read,result_rows,result_bytes,elapsed_ns,memory_usage,query_id
0,0,0,0,0,0,0,4792794,1114551,f0b7ce96-59b6-43b3-9c30-06909f604f57


read_rows,read_bytes,written_rows,written_bytes,total_rows_to_read,result_rows,result_bytes,elapsed_ns,memory_usage,query_id
0,0,0,0,0,0,0,5823056,1215303,e3b6c4f2-3e39-46a5-8d59-de3d910eac2b


The following code inserts the data into `data_source`.

In [29]:
--ClickHouse
INSERT INTO data_source VALUES 
    (1, 'target'),
    (2, 'some other'),
    (3, 'target');

The records with corresponding category appears in the corresponding `target_category` table.

In [30]:
--ClickHouse
SELECT * FROM target_category;

id,category
1,target
3,target
